### Unsupervised Learning

No labels. Two families: clustering (group similar points, K-Means/Hierarchical/DBSCAN/GMM) and dimensionality reduction (compress features while preserving structure, PCA/t-SNE/UMAP).

## Clustering: K-Means

#### 0. Core idea

Unsupervised, no labels. Group points into k clusters by minimizing within-cluster variance, sum of squared distances from each point to its cluster's centroid (mean).

Toy setup, 4 points, 2 obvious clusters, k=2:
```
(1, 1), (1.5, 2), (8, 8), (9, 9)
```
Initial centroids, picked as two of the points: c1=(1,1), c2=(9,9).


#### 1. The algorithm loop, worked by hand

Repeat: assign each point to its nearest centroid, recompute each centroid as the mean of its assigned points, until nothing changes.

Iteration 1, assign:
```
(1,1):   dist to c1=0.00,  dist to c2=11.31  -> cluster 1
(1.5,2): dist to c1=1.12,  dist to c2=10.26  -> cluster 1
(8,8):   dist to c1=9.90,  dist to c2=1.41   -> cluster 2
(9,9):   dist to c1=11.31, dist to c2=0.00   -> cluster 2
```
Recompute centroids:
```
new c1 = mean((1,1), (1.5,2)) = (1.25, 1.5)
new c2 = mean((8,8), (9,9))   = (8.5, 8.5)
```
Iteration 2, assign again with the new centroids:
```
(1,1):   dist to c1=0.56, dist to c2=huge -> cluster 1 (unchanged)
(1.5,2): dist to c1=0.56, dist to c2=huge -> cluster 1 (unchanged)
(8,8):   dist to c2=0.71 -> cluster 2 (unchanged)
(9,9):   dist to c2=0.71 -> cluster 2 (unchanged)
```
No point switched clusters, converged after just one centroid update. Final inertia (sum of squared distances to assigned centroid): cluster1 contributes 0.0625+0.25+0.0625+0.25=0.625, cluster2 contributes 0.25*4=1.0, total inertia = 1.625.


In [ ]:
import numpy as np

points = np.array([[1, 1], [1.5, 2], [8, 8], [9, 9]])
centroids = np.array([[1, 1], [9, 9]], dtype=float)

for iteration in range(5):
    distances = np.linalg.norm(points[:, None] - centroids[None, :], axis=2)
    assignments = np.argmin(distances, axis=1)
    new_centroids = np.array([points[assignments == k].mean(axis=0) for k in range(2)])
    print(f"iteration {iteration}: assignments={assignments}, centroids={new_centroids.tolist()}")
    if np.allclose(new_centroids, centroids):
        print("converged")
        break
    centroids = new_centroids

inertia = sum(np.sum((points[assignments == k] - centroids[k]) ** 2) for k in range(2))
print("\nfinal inertia:", inertia)

#### 2. Choosing k: the elbow method

Inertia always decreases as k increases (more clusters can only fit the data at least as well, in the extreme k=n_points gives inertia=0, every point is its own cluster, meaningless). Plot inertia against k, look for the "elbow", the point where adding another cluster stops giving a big drop in inertia and starts giving only marginal improvement. That elbow is the suggested k, a judgment call, not an exact formula.

#### 3. Limitations

- Sensitive to initialization: random starting centroids can converge to a bad local optimum, different runs can give different clusterings. k-means++ (sklearn's default init) picks initial centroids spread out from each other, reduces this risk but does not eliminate it, still worth running multiple times with different seeds (n_init) and keeping the lowest-inertia result.
- Assumes roughly spherical, similarly-sized clusters: it minimizes distance to a single centroid per cluster, so elongated or unevenly-sized clusters get split or merged incorrectly. DBSCAN (the density-based notebook in this series) does not share this assumption.
- k must be chosen upfront: unlike DBSCAN, which infers the number of clusters from density.
- Sensitive to feature scale: same reason as KNN, distance-based, unscaled features with large raw ranges dominate the distance calculation.


In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
kmeans.fit(points)
print("sklearn labels:", kmeans.labels_)
print("sklearn centroids:", kmeans.cluster_centers_)
print("sklearn inertia:", kmeans.inertia_)

# elbow method: inertia for a range of k
for k in range(1, 4):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(points)
    print(f"k={k}: inertia={km.inertia_:.3f}")

## Clustering: Hierarchical

#### 0. Core idea

Agglomerative (bottom-up): start with every point as its own cluster, repeatedly merge the two closest clusters, until everything is in one cluster. Builds a full tree (dendrogram) of merges, no need to pick k upfront, choose k afterward by cutting the tree at a chosen height.

Toy setup, 1D for simple distance math: points [1, 2, 5, 9].
Pairwise distances: |1-2|=1, |1-5|=4, |1-9|=8, |2-5|=3, |2-9|=7, |5-9|=4.


#### 1. Full agglomeration, worked step by step (single linkage: cluster distance = min distance between any pair of points across the two clusters)

```
Step 1: closest pair is (1,2), distance 1. Merge into A={1,2}.
        remaining points: 5, 9

Step 2: recompute distances from A to remaining points, using MIN:
        d(A,5) = min(|1-5|, |2-5|) = min(4,3) = 3
        d(A,9) = min(|1-9|, |2-9|) = min(8,7) = 7
        d(5,9) = 4
        closest pair now: (A,5), distance 3. Merge into B={1,2,5}.

Step 3: recompute distance from B to remaining point 9:
        d(B,9) = min(|1-9|, |2-9|, |5-9|) = min(8,7,4) = 4
        only one merge left: B and {9}, distance 4. Merge into {1,2,5,9}.
```
Merge order and heights: (1,2) at height 1, (A,5) at height 3, (B,9) at height 4. This sequence of heights IS the dendrogram.

Cutting the tree at height 2 (between 1 and 3): clusters = {1,2}, {5}, {9}, 3 clusters.
Cutting the tree at height 3.5 (between 3 and 4): clusters = {1,2,5}, {9}, 2 clusters.


In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

points = np.array([[1], [2], [5], [9]])

Z = linkage(points, method="single")
print("linkage matrix [idx1, idx2, distance, cluster_size]:\n", Z)

# cut the tree to get a specific number of clusters
clusters_3 = fcluster(Z, t=3, criterion="maxclust")
clusters_2 = fcluster(Z, t=2, criterion="maxclust")
print("\n3 clusters:", clusters_3)
print("2 clusters:", clusters_2)

#### 2. Linkage criteria change the answer

At the first merge step above (A={1,2} vs remaining points 5 and 9), compare how each linkage type would compute d(A,5):
```
single (min):    d(A,5) = min(|1-5|, |2-5|) = min(4,3) = 3
complete (max):  d(A,5) = max(4,3) = 4
average (mean):  d(A,5) = (4+3)/2 = 3.5
```
Single linkage tends to produce long, straggly clusters (chains together through nearest neighbors, can link two far-apart groups through a bridge of intermediate points). Complete linkage tends to produce tight, compact, evenly-sized clusters (forces the whole cluster to be close, not just one point). Average and Ward (minimizes variance increase, closest in spirit to k-means's objective) sit in between, Ward is the most commonly used default in practice for roughly this reason.

#### 3. Practical notes

No need to choose k upfront, decide after seeing the full dendrogram, a real advantage over k-means when the right number of clusters is not known in advance. Cost: O(n^2) or worse in the number of points, does not scale to large datasets the way k-means does. Still distance-based, feature scaling still matters, same reason as KNN and k-means.


In [ ]:
for method in ["single", "complete", "average", "ward"]:
    Z_method = linkage(points, method=method)
    print(f"{method} linkage, first merge distance:", Z_method[0, 2])

from sklearn.cluster import AgglomerativeClustering

model = AgglomerativeClustering(n_clusters=2, linkage="single")
labels = model.fit_predict(points)
print("\nsklearn AgglomerativeClustering, 2 clusters:", labels)

## Clustering: DBSCAN

#### 0. Core idea

Density-based, not centroid-based or hierarchical. Two parameters: eps (neighborhood radius), min_samples (how many points, including itself, must be within eps for a point to count as dense). Clusters grow outward from dense regions, points in sparse regions are labeled noise, not forced into any cluster. No need to choose k, the number of clusters falls out of the density structure.

Toy setup, 1D, eps=1, min_samples=3: points [1, 1.5, 2, 2.5, 3.4, 10].


#### 1. Classifying every point: core, border, noise

For each point, count neighbors (including itself) within eps=1:
```
x=1:    neighbors {1, 1.5, 2}           count=3  -> CORE (>= min_samples=3)
x=1.5:  neighbors {1, 1.5, 2, 2.5}      count=4  -> CORE
x=2:    neighbors {1, 1.5, 2, 2.5}      count=4  -> CORE
x=2.5:  neighbors {1.5, 2, 2.5, 3.4}    count=4  -> CORE
x=3.4:  neighbors {2.5, 3.4}            count=2  -> not core (below min_samples)
x=10:   neighbors {10}                  count=1  -> not core, and isolated
```
x=3.4 is not core itself, but it IS within eps of a core point (2.5, distance 0.9), so it gets pulled into that cluster as a BORDER point, on the edge, reachable but not dense enough to extend the cluster further.

x=10 is not core, and not within eps of any core point either (its nearest neighbor is 3.4, distance 6.6, way past eps=1). Labeled NOISE, correctly excluded from every cluster.

Final result: one cluster = {1, 1.5, 2, 2.5, 3.4} (4 core points plus 1 border point), and x=10 is noise, not forced into that cluster or any other. A k-means with k=1 or k=2 would have had to assign x=10 to SOME cluster, even though it is clearly an outlier, that forced assignment is exactly what DBSCAN avoids.


In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

points = np.array([[1], [1.5], [2], [2.5], [3.4], [10]])

db = DBSCAN(eps=1, min_samples=3)
labels = db.fit_predict(points)

print("labels (-1 means noise):", labels)
print("core sample indices:", db.core_sample_indices_)
for i, (pt, label) in enumerate(zip(points.flatten(), labels)):
    kind = "noise" if label == -1 else ("core" if i in db.core_sample_indices_ else "border")
    print(f"x={pt}: label={label}, kind={kind}")

#### 2. Choosing eps and min_samples

min_samples: rule of thumb, at least (number of dimensions + 1), higher for noisier data. Too low, almost everything looks dense, too few noise points get flagged. Too high, almost nothing looks dense, everything becomes noise.

eps: commonly chosen via a k-distance plot, compute each point's distance to its k-th nearest neighbor (k=min_samples), sort ascending, look for the elbow, same visual idea as k-means's inertia elbow, just applied to neighbor distance instead.

#### 3. When DBSCAN wins over k-means

- Non-spherical clusters: DBSCAN follows density, can trace out arbitrary shapes (crescents, rings), k-means always carves spherical regions around a centroid regardless of the true shape.
- Outliers: DBSCAN labels them noise explicitly, k-means forces every point into the nearest cluster whether it belongs or not.
- No need to specify k: inferred from the data's density structure.
- Where it loses: clusters of very different densities in the same dataset are hard for one global eps to handle, and it does not scale as well as k-means on very large datasets (though tree-based neighbor search helps).


## Gaussian Mixture Models (GMM)

Soft, probabilistic clustering, unlike K-Means's hard assignment (every point belongs to exactly one cluster). Each cluster is modeled as a Gaussian distribution (mean, variance, and a mixing weight), and each point gets a PROBABILITY of belonging to each cluster, not a single hard label.

#### 0. Core idea: the EM algorithm

Two alternating steps, repeated until convergence:
- E-step (Expectation): given the current cluster parameters, compute each point's responsibility, the probability it belongs to each cluster.
- M-step (Maximization): given those responsibilities, update each cluster's mean, variance, and mixing weight to better fit the (now probabilistically labeled) data.

Toy setup, 1D, same 4 points as the K-Means/DBSCAN toys, plus one new point placed deliberately between the two clusters: [1, 2, 8, 9, 5]. Initial guess (before any E-step): 2 clusters, mu1=1.5, mu2=8.5, equal variance sigma^2=1, equal mixing weight pi1=pi2=0.5.

#### 1. E-step, worked by hand: responsibility of the ambiguous point x=5

Gaussian density: N(x|mu,sigma^2) = (1/sqrt(2*pi*sigma^2)) * exp(-(x-mu)^2 / (2*sigma^2))
```
N(5 | mu1=1.5, sigma=1) = 0.3989 * exp(-(3.5)^2/2) = 0.3989 * 0.00219 = 0.000874
N(5 | mu2=8.5, sigma=1) = 0.3989 * exp(-(3.5)^2/2) = 0.000874   (identical, x=5 is equidistant)
```
Responsibility (E-step), normalize the weighted densities:
```
r1 = (pi1 * N1) / (pi1*N1 + pi2*N2) = (0.5*0.000874) / (0.5*0.000874 + 0.5*0.000874) = 0.5
r2 = 0.5
```
x=5 gets EXACTLY 50% responsibility to each cluster, a genuinely fractional soft assignment. K-Means could never express this, it would force x=5 into whichever cluster's centroid happens to be marginally closer (or a coin-flip tie-break), losing the fact that this point is genuinely ambiguous. Now check a non-ambiguous point, x=2 (clearly near cluster 1):
```
N(2|mu1=1.5) = 0.3989*exp(-(0.5)^2/2) = 0.3521
N(2|mu2=8.5) = 0.3989*exp(-(6.5)^2/2) ~ 0 (essentially zero, far tail of the Gaussian)
r1 ~ 1.0, r2 ~ 0.0
```
When a point is genuinely close to one cluster, the soft responsibility collapses to nearly the same hard assignment K-Means would give, GMM's flexibility shows up specifically for ambiguous points like x=5, not for clear-cut ones.

In [ ]:
import numpy as np
from scipy.stats import norm

points = np.array([1, 2, 8, 9, 5])
mu1, mu2, sigma = 1.5, 8.5, 1.0
pi1, pi2 = 0.5, 0.5

def responsibility(x, mu1, mu2, sigma, pi1, pi2):
    n1 = norm.pdf(x, mu1, sigma)
    n2 = norm.pdf(x, mu2, sigma)
    r1 = (pi1 * n1) / (pi1 * n1 + pi2 * n2)
    return r1, 1 - r1

for x in points:
    r1, r2 = responsibility(x, mu1, mu2, sigma, pi1, pi2)
    print(f"x={x}: r(cluster1)={r1:.3f}, r(cluster2)={r2:.3f}")

#### 2. Practical notes

Requires choosing the number of components (like K-Means's k), same elbow-style model-selection problem, commonly via BIC/AIC instead of inertia. Assumes clusters are Gaussian-shaped (elliptical, can have different sizes/orientations via the covariance matrix, more flexible than K-Means's implicit spherical assumption, but still a parametric shape assumption, not fully shape-free like DBSCAN). Useful whenever "how confident is this assignment" matters as much as the assignment itself, e.g. flagging genuinely ambiguous cases for human review rather than silently forcing a hard label.

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(points.reshape(-1, 1))

probs = gmm.predict_proba(points.reshape(-1, 1))
for x, p in zip(points, probs):
    print(f"x={x}: cluster probabilities = {p.round(3)}")
print("\nhard labels (argmax of probabilities):", gmm.predict(points.reshape(-1, 1)))

## Dimensionality Reduction: PCA

#### 0. Core idea

Dimensionality reduction. Find new axes (principal components), each one a linear combination of the original features, ordered so PC1 captures the most variance in the data, PC2 the most of what is left after removing PC1, and so on, all mutually orthogonal (perpendicular, uncorrelated with each other).

Toy setup, 2D, roughly diagonal spread with some noise:
```
points: (1,1), (2,2), (3,3), (1,2)
```


#### 1. Center the data, compute the covariance matrix

Step 1, center: subtract the mean from every point so the data is centered at the origin (PCA is about variance/spread, not location).
```
mean = (1.75, 2.0)
centered points: (-0.75,-1), (0.25,0), (1.25,1), (-0.75,0)
```
Step 2, covariance matrix (sample covariance, divide by n-1=3):
```
Cov(x,x) = sum(x_i^2)/3 = (0.5625+0.0625+1.5625+0.5625)/3 = 0.917
Cov(y,y) = sum(y_i^2)/3 = (1+0+1+0)/3 = 0.667
Cov(x,y) = sum(x_i*y_i)/3 = (0.75+0+1.25+0)/3 = 0.667

C = [[0.917, 0.667],
     [0.667, 0.667]]
```
Cov(x,y)=0.667 is positive and fairly large relative to the variances, x and y move together, consistent with the roughly diagonal shape of the data.


#### 2. Eigenvalues and eigenvectors of the covariance matrix

Solve det(C - lambda*I) = 0:
```
(0.917-lambda)(0.667-lambda) - 0.667^2 = 0
lambda^2 - 1.583*lambda + 0.167 = 0
lambda = [1.583 +- sqrt(1.583^2 - 4*0.167)] / 2 = [1.583 +- 1.357] / 2

lambda1 = 1.470   (PC1's variance)
lambda2 = 0.113   (PC2's variance)
```
Check: lambda1+lambda2 should equal the trace (0.917+0.667=1.583, matches). lambda1*lambda2 should equal the determinant (0.917*0.667-0.667^2=0.167, matches).

Eigenvector for lambda1 (solve (C-lambda1*I)v=0, then normalize to unit length): v ~ (0.770, 0.639). This is the direction of PC1, close to the 45 degree diagonal (0.707, 0.707) but tilted slightly toward x, matching that x had higher variance (0.917) than y (0.667) in the raw data.

Explained variance ratio: PC1 captures lambda1/(lambda1+lambda2) = 1.470/1.583 = 0.929, about 93% of the total variance in just one dimension instead of two.


In [ ]:
import numpy as np

points = np.array([[1, 1], [2, 2], [3, 3], [1, 2]], dtype=float)
mean = points.mean(axis=0)
centered = points - mean

cov = np.cov(centered.T)  # bias=False by default, matches the n-1 division used above
print("covariance matrix:\n", cov)

eigenvalues, eigenvectors = np.linalg.eig(cov)
order = np.argsort(-eigenvalues)  # descending, PC1 first
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]

print("\neigenvalues:", eigenvalues)
print("PC1 direction:", eigenvectors[:, 0])
print("explained variance ratio:", eigenvalues / eigenvalues.sum())

# project onto PC1 only, this is the dimensionality reduction step
projected_1d = centered @ eigenvectors[:, 0]
print("\ndata projected onto PC1:", projected_1d)

#### 3. Standardize features first

PCA maximizes variance, and variance is scale-dependent, a feature measured in thousands (amount_lost) will dominate the covariance matrix over a feature measured in 0/1 (urgency_language) purely because of units, not because it is actually more informative. Standardizing every feature to mean 0, variance 1 before running PCA puts them on equal footing, without this, PCA effectively just rediscovers "which feature had the biggest raw scale," not the genuinely dominant pattern in the data.

#### 4. Practical notes

Components are ordered by variance explained, keep only the first few that together explain enough of the total (a common threshold: 90-95%). Components are linear combinations of ALL original features, not a subset, so unlike a tree's feature importance, you cannot point to "this one original feature drove the prediction," interpretability is traded for compression. Commonly used as a preprocessing step before a distance-based method (KNN, k-means) on high-dimensional data, since it both reduces the curse-of-dimensionality problem and removes correlated redundancy.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
projected_sklearn = pca.fit_transform(points)

print("sklearn PC1 direction:", pca.components_[0])
print("sklearn explained variance ratio:", pca.explained_variance_ratio_)
print("sklearn projection:", projected_sklearn.flatten())
print("\n(sign of the component can flip vs. our manual version, direction is the same line, just possibly pointing the opposite way, that is expected and not an error)")

## t-SNE and UMAP

Nonlinear dimensionality reduction, unlike PCA's linear projection (a straight-line change of axes). Built specifically for visualization, preserving local neighborhood structure (which points are close to which) rather than PCA's global variance-maximizing objective. Cannot hand-compute a full embedding (it is an iterative gradient descent optimization), but the core mechanism, converting distances into neighbor probabilities, is hand-computable and is the actual conceptual heart of the method.

#### 0. t-SNE core idea: distances become neighbor probabilities

In the original high-dimensional space, convert each point's distances to every other point into a probability distribution over "which point is my neighbor", using a Gaussian kernel centered at that point:
```
p(j|i) = exp(-||x_i - x_j||^2 / (2*sigma_i^2)) / sum_k exp(-||x_i - x_k||^2 / (2*sigma_i^2))
```
sigma_i is tuned per point (not a single global value) to hit a target perplexity, a knob controlling the effective number of neighbors considered, small perplexity focuses on very local structure, large perplexity considers a broader neighborhood.

Worked example, 3 points in 1D, sigma=1 fixed for simplicity: x1=1, x2=2, x3=10 (two close together, one far away). Compute point 1's neighbor distribution:
```
dist(x1,x2)^2 = 1,  dist(x1,x3)^2 = 81

exp(-1/2)  = 0.6065
exp(-81/2) = exp(-40.5) ~ 0  (essentially zero, the exponential decays extremely fast)

p(2|1) = 0.6065 / (0.6065 + ~0) ~ 1.0
p(3|1) = ~0.0
```
Point 1's "neighbor probability" is essentially entirely concentrated on point 2 (its true near neighbor), point 3 contributes almost nothing despite being a real point in the dataset. This is the mechanism: convert raw distance into a distribution that sharply favors close points and nearly ignores far ones.

In [ ]:
import numpy as np

points = np.array([1, 2, 10])
sigma = 1.0

def neighbor_probs(i, points, sigma):
    dists_sq = (points - points[i]) ** 2
    weights = np.exp(-dists_sq / (2 * sigma ** 2))
    weights[i] = 0  # a point is not its own neighbor
    return weights / weights.sum()

for i in range(len(points)):
    probs = neighbor_probs(i, points, sigma)
    print(f"point {i} (x={points[i]}): neighbor probabilities = {probs.round(4)}")

#### 1. The rest of the mechanism, conceptually

Step 2: do the same thing in the LOW-dimensional embedding space (the 2D or 3D output being optimized), but using a heavier-tailed Student-t distribution instead of Gaussian, this is t-SNE's fix for the "crowding problem", in low dimensions there simply is not enough room to preserve every high-dimensional distance relationship, a heavier tail lets moderately-distant points in the embedding stay further apart than a Gaussian would allow, reducing the tendency for everything to collapse toward the center.

Step 3: minimize KL divergence (see `loss-functions.ipynb`) between the high-dimensional neighbor-probability distribution and the low-dimensional one, via gradient descent on the embedding POINTS themselves (not model weights, the coordinates being optimized ARE the output).

#### 2. UMAP, briefly

Same overall goal (nonlinear, neighborhood-preserving dimensionality reduction), different mathematical foundation, based on fuzzy topological structure and manifold theory rather than a Gaussian-vs-Student-t probability matching scheme. Practically: generally faster than t-SNE on large datasets, and tends to preserve more of the global structure (relative distances between distant clusters, not just which points are local neighbors) that t-SNE is known to distort.

#### 3. Practical notes

Both are for VISUALIZATION and exploration primarily, not a general-purpose preprocessing step before another model the way PCA often is, the low-dimensional coordinates do not have a stable, reusable meaning across different runs or different data (rerunning with a different random seed can produce a visually different but equally valid layout). Distances between well-separated clusters in a t-SNE plot are not reliably meaningful, only the local neighborhood structure within a cluster is trustworthy, a common misreading of these plots.

In [ ]:
from sklearn.manifold import TSNE
import numpy as np

rng = np.random.default_rng(0)
cluster_a = rng.normal(loc=[0, 0], scale=0.5, size=(15, 2))
cluster_b = rng.normal(loc=[10, 10], scale=0.5, size=(15, 2))
X = np.vstack([cluster_a, cluster_b])

tsne = TSNE(n_components=2, perplexity=5, random_state=42)
embedded = tsne.fit_transform(X)
print("embedded shape:", embedded.shape)
print("first 3 embedded points:\n", embedded[:3].round(2))

# UMAP: pip install umap-learn (not in this environment)
# from umap import UMAP
# umap_embedded = UMAP(n_components=2, random_state=42).fit_transform(X)